# The regularized greedy operator, worked by hand

This notebook walks the identity at the heart of Article 1 §3 of the [RLVR Operator Series](https://github.com/lmdixon23/rlvr-operator-series):

$$\pi^\star \;=\; \nabla \Omega^\ast(q), \qquad \Omega^\ast(q) \;=\; \max_{\pi \in \Delta}\;\langle \pi, q\rangle - \Omega(\pi).$$

We will:
1. Fence off the most common misconception about this identity.
2. Work the entropy-regularized case by hand on a 3-action example.
3. Verify against `operator_zoo.RegularizedOp`.
4. Repeat with the KL-anchored (Vieillard) operator.
5. Show why a GRPO-style group-normalized advantage plugs in as `q`.

## The advantage is `q`, not `Ω`

Article 1 §3 spells this out explicitly:

> *"When we reach GRPO, its group normalized advantage is not the regularizer at all, it is the linear score `q` fed into the step, while the regularizer remains the entropy or KL penalty that softens the maximum."*

Why does this trip people up? Because a reader coming from deep RL has seen "advantage" used as a coefficient on a gradient term, and the variational form `⟨π, q⟩ - Ω(π)` *also* puts the advantage-like quantity in a linear slot. The temptation is to identify the two and conclude that "GRPO's advantage **is** the regularizer." It is not. The two slots have completely different roles:

| Slot         | What it is                          | Examples                                                           |
|--------------|-------------------------------------|--------------------------------------------------------------------|
| `q` (score)  | Linear pressure toward higher value | action values, group-normalized advantages, learned token rewards  |
| `Ω` (regularizer) | Convex penalty that softens the max | negative entropy, KL-to-uniform, KL-to-anchor                      |

When we substitute the group-normalized advantage `A_i = (R_i - mean(R)) / std(R)` into the operator, it goes into the `q` slot. The regularizer `Ω` is still entropy or a KL anchor. That is what Article 1 means by "GRPO methods are knobs on a single skeleton" — the *score* knob is `q`, the *regularizer* knob is `Ω`, and they are not the same thing.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
from operator_zoo import RegularizedOp

## Entropy-regularized: by hand

Take `q = [1.0, 2.0, 0.5]` and `β = 1`. The negative-entropy regularizer is `Ω(π) = Σ π log π`. The maximizer of `⟨π, q⟩ - β·Ω(π)` is the softmax of `q / β`. The regularized value is `Ω*(q) = β log Σ exp(q/β)` — the soft maximum.

Let us compute both by hand.

In [ ]:
q = np.array([1.0, 2.0, 0.5])
beta = 1.0

# Step 1: softmax of q/beta — this is the maximizing policy.
ez = np.exp(q / beta)
pi_hand = ez / ez.sum()

# Step 2: the soft maximum — this is Ω*(q).
Omega_star_hand = beta * np.log(ez.sum())

# Step 3: the penalty Ω(π) evaluated at the maximizing policy.
Omega_at_pi_hand = beta * np.sum(pi_hand * np.log(pi_hand))

print(f'π★          = {pi_hand}')
print(f'Ω*(q)        = {Omega_star_hand:.6f}')
print(f'Ω(π★)        = {Omega_at_pi_hand:.6f}')
print(f'⟨π★, q⟩      = {float(pi_hand @ q):.6f}')
print(f'⟨π★, q⟩ - Ω(π★) = {float(pi_hand @ q) - Omega_at_pi_hand:.6f}   (should equal Ω*(q))')

## Verify against the zoo

Two identities to confirm:

1. **`Ω*(q) = ⟨π★, q⟩ - Ω(π★)`** (the easy one).
2. **`π★ = ∇Ω*(q)`** (the harder, more fundamental one — verified here by finite difference).

Note the use of the `tau` alias on `NegativeEntropy` — it constructs the same operator as `beta`, but it is the symbol the series bible names for the entropy coefficient (Geist, Vieillard).

In [ ]:
# Construct with the bible-aligned symbol.
op = RegularizedOp('entropy', tau=1.0)

pi_zoo  = op.policy(q)
Omega_star_zoo = op.value(q)

# Identity 1.
rhs = float(pi_zoo @ q) - op.omega(pi_zoo)
print(f'Ω*(q) - (⟨π,q⟩ - Ω(π)) = {Omega_star_zoo - rhs:.2e}   (should be ~1e-15)')

# Identity 2 via finite difference of Ω*.
eps = 1e-6
grad_Omega_star = np.zeros_like(q)
for i in range(q.size):
    q_plus  = q.copy();  q_plus[i]  += eps
    q_minus = q.copy();  q_minus[i] -= eps
    grad_Omega_star[i] = (op.value(q_plus) - op.value(q_minus)) / (2 * eps)
print(f'∥π★ - ∇Ω*(q)∥ = {np.linalg.norm(pi_zoo - grad_Omega_star):.2e}   (should be ~1e-6)')

## KL-anchored (Vieillard): closed form

Now the regularizer is `Ω(π) = λ·KL(π ∥ μ)` for a reference (anchor) policy `μ`. The closed-form maximizer is

$$\pi^\star(a) \;\propto\; \mu(a)\;\exp(q(a)\,/\,\lambda).$$

This is the Vieillard (2020) anchored-tilt identity. Note the use of `lambda_kl` — the bible's name for Vieillard's KL coefficient. The `_kl` suffix disambiguates it from `lambda_trace`, GRPO-lambda's eligibility-trace parameter (the worst notation collision the bible warns about).

In [ ]:
mu = np.array([0.6, 0.3, 0.1])
lam = 1.5

# By hand: μ · exp(q/λ), then normalize.
unnorm = mu * np.exp(q / lam)
pi_hand_kl = unnorm / unnorm.sum()

# By the zoo, with the bible-aligned `lambda_kl` alias.
pi_zoo_kl = RegularizedOp('kl_anchor', lambda_kl=lam, anchor=mu).policy(q)

print(f'π★ (hand)  = {pi_hand_kl}')
print(f'π★ (zoo)   = {pi_zoo_kl}')
print(f'∥diff∥      = {np.linalg.norm(pi_hand_kl - pi_zoo_kl):.2e}   (should be ~1e-15)')

## Tying back to GRPO

GRPO samples a group of `G` completions for a prompt, scores each with a verifier, and computes the group-normalized advantage

$$A_i \;=\; \frac{R_i - \mathrm{mean}(R)}{\mathrm{std}(R)}.$$

When the variational view says "GRPO is an instance of the regularized greedy step," the substitution is:

$$q \;\leftarrow\; A \;=\; (A_1, \dots, A_G).$$

Crucially, `Ω` is still the same KL-to-`π_ref` penalty that appears in GRPO's loss; it is not replaced by `A`. Below: feed a synthetic group of advantages into the KL-anchored operator with a uniform anchor — the resulting policy concentrates probability on the high-advantage responses, which is exactly the GRPO update direction.

In [ ]:
# Five sampled responses, three of which were correct.
R = np.array([1.0, 0.0, 1.0, 1.0, 0.0])
A = (R - R.mean()) / (R.std() + 1e-12)
print(f'Advantages A = {A}')

# Feed A into the score slot; anchor to the uniform 'old' policy.
mu_old = np.full_like(A, 1.0 / A.size)
pi_new = RegularizedOp('kl_anchor', lambda_kl=0.5, anchor=mu_old).policy(A)

print(f'Old policy   = {mu_old}')
print(f'New policy   = {np.round(pi_new, 3)}')
print(f'\\nResponses with advantage > 0 gained probability;')
print(f'those with advantage < 0 lost it. The regularizer Ω (the KL')
print(f'anchor) is unchanged. That is the GRPO update in one identity.')

## Summary

- The maximizer `π★` of `⟨π, q⟩ - Ω(π)` is the gradient `∇Ω*(q)` of the convex conjugate. We verified this numerically by finite difference.
- The conjugate identity `Ω*(q) = ⟨π★, q⟩ - Ω(π★)` is the algebraic consequence and is easier to check.
- GRPO substitutes the group-normalized advantage into the `q` slot. The regularizer `Ω` is still entropy or a KL anchor; it is not the advantage.
- The bible-aligned aliases (`tau`, `lambda_kl`) are interchangeable with `beta` at the API level; pick the symbol that matches the article you're reading from.

Next: see `notebooks/policy_vs_beta.png` (produced by `python -m smoke.run_smoke`) for the same identity exercised across four operators as `β` sweeps from 0.01 to 100.